# Fabric Data Warehousing

[Microsoft Fabric Data Warehouse [Full Course] | Future Of Azure Data Engineering](https://www.youtube.com/watch?v=sgvbLKomAuk&t=1806s)

## Data Lake present in form of OneLake
- ETL
  - Data Pipelines - to load the data
  - Data flows - are light weight pipelines
- Copy into Command >  convert to delta parquet
- CTAS (Polybase)

In [0]:
## Openrowset
---
View the data from Azure storage
---

SELECT *
FROM
OPENROWSET
(
  BULK 'https://URL for file in azure storage account'
  FORMAT = 'Parquet'
) query


- Give access to User for each Role: StorageBlobDataContributor
- Does not work for tables in OneLake

In [0]:
## Copy Into
---
Load the data from Azure storage to OneLake
---
CREATE TABLE dbo.orders
(
  order_id VARCHAR(255),
  customer_id VARCHAR(255),
  product_id VARCHAR(255),
  orderdate DATE,
  quantity INT,
  unitprice DECIMAL(16, 2),
  totalamount DECIMAL (16, 2)
)

COPY INTO dbo.orders
FROM 'https://URL for file in azure storage account'
WITH
(
  FILE_TYPE = 'Parquet'
)

select * from dbo.orders

In [0]:
## CTAS
---
Create Table as Select
---
CREATE TABLE dbo.products
AS
SELECT * FROM
OPENROWSET
(
  BULK 'https://URL for file in azure storage account'
  FORMAT = 'Parquet'
) query


CREATE TABLE dbo.products_new
AS
SELECT * FROM dbo.products
where customer_id = "1002"

In [0]:
## Data Pipelines

Activities > Copy Data > Linked Service
- Azure Data Lake Storage Gen2
   - Storage account > Endpoint > https://...
- Online Data Source

Destination > FabricWarehouse


Cloning and Time Travel

In [0]:
-- Create clone at a specific point in time
CREATE TABLE dbo.ordersclone
AS
CLONE OF dbo.orders
AT '2025-01-02'

-- State of a Table at a particular TimeStamp
SELECT * FROM dbo.demo
OPTION (FOR TIMESTAMP AS OF '2025-01-02')



## Stored Procedures
Similar to a function in Python

In [0]:
CREATE PROCEDURE dbo.sp_demo
AS
  BEGIN
    DROP TABLE IF EXISTS dbo.demoSP;
    CREATE TABLE dbo.demoSP
    (
      id INT,
      name VARCHAR(255)
    );
    INSERT INTO dbo.demoSP
    VALUES (1, 'John'), (2, 'Jane');
  END;

EXEC dbo.sp_demo;

-- Watermarking table
-- update a column in the watermarked table

## Schemas

In [0]:
-- sys schema

IF NOT EXISTS (SELECT * FROM sys.schemas s JOIN sys.tables t ON s.schema_id = t.schema_id WHERE s.name = 'dbo' AND t.name = 'orders')
BEGIN
  CREATE TABLE dbo.orders
  (
    id INT
  )
END;

In [0]:
CREATE SCHEMA silver
GO;

-- create views for enriched silver layer
-- Optional: Can use Visual Query editor
-- Sorting does not work to save as view

CREATE SCHEMA gold;

CREATE TABLE gold.orders
AS
SELECT *
FROM silver.orders_view

In [0]:
-- create business views from gold layer
-- CTE: Common Table Expression
-- nested CTEs cannot be used for views

with caterorical_sales as (
SELECT 
  order_id, p.category
  SUM(totalamount) as sales
FROM gold.orders o
JOIN silver.products p ON o.product_id = p.id
GROUP BY order_id, p.category
)
SELECT *
FROM caterorical_sales;

with order_count as (
SELECT 
  order_id, p.category
FROM gold.orders o
JOIN silver.products p ON o.product_id = p.id
)
SELECT category, count(order_id) as total_orders
FROM order_count
GROUP BY category

-- multiple CTEs
with oneCTE AS (),
twoCTE AS ()
SELECT *
FROM oneCTE
UNION ALL
SELECT *
FROM twoCTE

-- window function for ranking

## Semantic model
Serve the data to other users